In [ ]:
import torch

import sys
sys.path.append('..')

from lucid.geometry import generate_detector
from lucid.utils import load_single_event, save_single_event, generate_random_params, print_particle_params
from lucid.losses import WC_loss
from lucid.simulation import setup_event_simulator
from lucid.generate import read_photon_data_from_photonsim
from lucid.detector_params import ParticleParams, load_detector_params

import jax
import jax.numpy as jnp
import time

from jax import jit
from pathlib import Path

from matplotlib import pyplot as plt
plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 10

import numpy as np
from functools import partial
import pickle
from tqdm import tqdm
from jax import grad, jit, vmap, value_and_grad
import uproot

In [ ]:
default_json_filename = '../config/SK_geom_config.json'
data_file = '../data/water/muon/muon_gun_1050_MeV_100_events_fixed_energy.root'

PHYSICS_CONFIG = '../config/SK_physics_config.json'
detector_params = load_detector_params(PHYSICS_CONFIG)

detector = generate_detector(default_json_filename)
detector_points = jnp.array(detector.all_points)
detector_radius = detector.S_radius
NUM_DETECTORS = len(detector_points)
Nphot = 1_000_000

K=20

prediction_simulator = setup_event_simulator(default_json_filename, Nphot, temperature=0.1, K=7, is_data=False, hit_mode='aggregated', physics_config=PHYSICS_CONFIG, default_detector_params=True)

data_simulator = setup_event_simulator(default_json_filename, Nphot, temperature=0.0, K=K,
                                      is_data=True, is_calibration=False, physics_config=PHYSICS_CONFIG, default_detector_params=True)

print(f"Number of detectors: {NUM_DETECTORS}")

# Load ROOT file information
with uproot.open(data_file) as file:
    tree = file['OpticalPhotons']
    n_entries = tree.num_entries
print(f"ROOT file has {n_entries} entries")

In [ ]:
# Generate random entry index and event parameters
key = jax.random.PRNGKey(719007)
entry_key = jax.random.PRNGKey(12345)

# Select random entry from ROOT file
entry_idx = 4#int(jax.random.randint(entry_key, shape=(), minval=0, maxval=n_entries))
print(f"Loading entry {entry_idx} from ROOT file")

# Load photon data from ROOT file
photon_data = read_photon_data_from_photonsim(data_file, entry_idx)
photon_data['N'] = len(photon_data['photon_origins'])

# Generate track parameters (position and direction)
# Use detector bounds for random position generation
detector_bounds = {
    'type': 'cylinder',
    'r': detector.r,
    'H': detector.H
}

true_position = jnp.array([-10., 0., 0.])
true_direction = jnp.array([1., 0., 0.])

# Use energy from ROOT file
true_energy = photon_data['energy']

# Create particle parameters
true_track = ParticleParams.from_cartesian(
    energy=true_energy, position=true_position,
    direction=true_direction, t0=0.0
)

original_direction = jnp.array([0.0, 0.0, 1.0])
true_direction_norm = true_direction / (jnp.linalg.norm(true_direction) + 1e-8)

# Rotation axis = cross product of original and target directions
rotation_axis = jnp.cross(original_direction, true_direction_norm)
axis_norm = jnp.linalg.norm(rotation_axis)

# Handle case where directions are parallel (axis_norm ~ 0)
rotation_axis = jnp.where(
    axis_norm < 1e-6,
    jnp.array([1.0, 0.0, 0.0]),  # Arbitrary axis when parallel
    rotation_axis / (axis_norm + 1e-8)
)

# Rotation angle = arccos of dot product
rotation_angle = jnp.arccos(jnp.clip(
    jnp.dot(original_direction, true_direction_norm), -1.0, 1.0
))

# Set rotation parameters
photon_data['rotation_axis'] = rotation_axis
photon_data['rotation_angle'] = rotation_angle
photon_data['apply_rotation'] = jnp.array(True)

# Set translation parameters to move from origin to true_position
photon_data['apply_translation'] = jnp.array(True)
photon_data['translation_vector'] = true_position

true_data = jax.lax.stop_gradient(data_simulator(true_track, key, photon_data))

pred_data = jax.lax.stop_gradient(prediction_simulator(true_track, key))

print("True parameters:")
print(f"  Energy: {true_energy:.2f} MeV")
print(f"  Position: [{true_position[0]:.2f}, {true_position[1]:.2f}, {true_position[2]:.2f}] m")
print(f"  Direction: [{true_direction[0]:.3f}, {true_direction[1]:.3f}, {true_direction[2]:.3f}]")
print(f"  Photon data: {photon_data['N']} photons")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import EllipseCollection
from scipy.spatial.distance import pdist
from mpl_toolkits.axes_grid1 import make_axes_locatable
from lucid.geometry import generate_detector
from lucid.utils import sparse_to_full
from matplotlib.colors import LinearSegmentedColormap

def create_color_gradient(max_cnts, colormap='gnuplot'):
    cmap = plt.get_cmap(colormap)
    norm = plt.Normalize(vmin=0, vmax=max_cnts)
    return plt.cm.ScalarMappable(norm=norm, cmap=cmap)


def calculate_min_distance(positions):
    distances = pdist(positions)
    return np.min(distances) if len(distances) > 0 else 1.0


def create_detector_display(json_filename='../config/cyl_geom_config.json', sparse=True):
    """
    Create a detector display function that can handle both sparse and dense data formats.

    Parameters:
    -----------
    json_filename : str
        Path to the configuration file for detector geometry
    sparse : bool
        If True, function expects sparse data format (indices, charges, times)
        If False, function expects dense data format (full arrays)

    Returns:
    --------
    function
        Display function that can be called with appropriate data format
    """
    # Generate detector
    detector = generate_detector(json_filename)
    radius = detector.r
    height = detector.H
    sensor_radius = detector.S_radius

    # Set up detector information
    sensor_positions = np.array(detector.all_points)
    sensor_cases = np.array([detector.ID_to_case[i] for i in range(len(detector.all_points))])
    n_sensors = len(sensor_positions)

    def display_detector_data(*args, file_name=None, plot_time=False, log_scale=False, vmin=None, vmax=None, perc_min=1,
                              perc_max=99):
        """
        Process and display detector data in either sparse or dense format.

        Parameters (for sparse=True):
        ---------------------------
        loaded_indices : array-like
            Indices of non-zero hits
        loaded_charges : array-like
            Charge values at non-zero indices
        loaded_times : array-like
            Time values at non-zero indices

        Parameters (for sparse=False):
        ---------------------------
        charges : array-like
            Full array of charge values
        times : array-like
            Full array of time values

        Other Parameters:
        ----------------
        file_name : str, optional
            If provided, saves the plot to this file
        plot_time : bool
            If True, plot time instead of charge
        log_scale : bool
            If True, apply logarithmic scaling to the color gradient
        vmin : float, optional
            Minimum value for colormap. Values below this but > 0 will be shown in a distinct color
        vmax : float, optional
            Maximum value for colormap
        perc_min : float
            Percentile to use for minimum if vmin not provided (default: 1)
        perc_max : float
            Percentile to use for maximum if vmax not provided (default: 99)
        """
        if sparse:
            if len(args) != 3:
                raise ValueError("Sparse format requires three arguments: indices, charges, and times")
            loaded_indices, loaded_charges, loaded_times = args

            # Convert sparse to full arrays
            all_charges = sparse_to_full(loaded_indices, loaded_charges, n_sensors)
            all_times = sparse_to_full(loaded_indices, loaded_times, n_sensors)
        else:
            if len(args) != 2:
                raise ValueError("Dense format requires two arguments: charges and times arrays")
            all_charges, all_times = args

        # Select which values to plot based on plot_time
        all_values = all_times if plot_time else all_charges

        # Get positive values for percentile calculations
        positive_values = all_values[all_values > 0]

        # Calculate percentiles if vmin or vmax not provided
        if len(positive_values) > 0:
            if vmin is None:
                vmin = np.percentile(positive_values, perc_min)
            if vmax is None:
                vmax = np.percentile(positive_values, perc_max)
        else:
            # Fallback if no positive values
            if vmin is None:
                vmin = 0.1 if log_scale else 0
            if vmax is None:
                vmax = 1

        if perc_min == 0 and plot_time == False and log_scale == False:
            vmin = 0.001
        elif perc_min == 0 and plot_time == False and log_scale == True:
            vmin = 0.1

        facecolor='white'
        barcolor='black'

        # Generate color gradient based on scale type
        max_value = vmax

        if log_scale:
            # Create a copy of values for color mapping
            plot_values = np.copy(all_values)
            # Clip values below vmin to vmin (instead of setting to transparent)
            plot_values[plot_values <= 0] = vmin
            plot_values = np.clip(plot_values, vmin, max_value)

            # Use LogNorm for logarithmic color scaling
            cmap = plt.get_cmap('viridis_r') if plot_time else plt.get_cmap('plasma')
            norm = plt.matplotlib.colors.LogNorm(vmin=vmin, vmax=max_value)
            color_gradient = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
        else:
            # Linear scaling
            plot_values = np.copy(all_values)
            # Clip values to the range [vmin, vmax]
            plot_values = np.clip(plot_values, vmin, max_value)

            cmap = plt.get_cmap('viridis_r') if plot_time else plt.get_cmap('viridis')
            norm = plt.Normalize(vmin=vmin, vmax=max_value)
            color_gradient = plt.cm.ScalarMappable(norm=norm, cmap=cmap)

        caps_offset = 1.05 * height / 2 + radius

        # Calculate positions for all cases
        x = np.zeros(n_sensors)
        y = np.zeros(n_sensors)

        # Barrel case (0)
        barrel_mask = sensor_cases == 0
        theta = np.arctan2(sensor_positions[barrel_mask, 1], sensor_positions[barrel_mask, 0])
        theta = (theta + np.pi * 3 / 2) % (2 * np.pi) / 2
        x[barrel_mask] = theta * radius * 2
        y[barrel_mask] = sensor_positions[barrel_mask, 2]

        # Top cap case (1)
        top_mask = sensor_cases == 1
        x[top_mask] = sensor_positions[top_mask, 0] + np.pi * radius
        y[top_mask] = (caps_offset + sensor_positions[top_mask, 1])

        # Bottom cap case (2)
        bottom_mask = sensor_cases == 2
        x[bottom_mask] = sensor_positions[bottom_mask, 0] + np.pi * radius
        y[bottom_mask] = (-caps_offset - sensor_positions[bottom_mask, 1])

        # Calculate the minimum distance between points in the transformed space
        transformed_positions = np.column_stack((x, y))
        min_distance = calculate_min_distance(transformed_positions)

        # Set the circle diameter to be equal to the minimum distance
        circle_diameter = min_distance

        # Calculate exact dimensions needed
        x_min, x_max = np.min(x), np.max(x)
        y_min, y_max = np.min(y), np.max(y)

        # Add padding
        padding = circle_diameter
        x_min -= padding
        x_max += padding
        y_min -= padding
        y_max += padding
        x_range = x_max - x_min
        y_range = y_max - y_min

        # Set figure size based on data range, accounting for colorbar
        fig_width = 8
        fig_height = fig_width * (y_range / x_range)

        fig, ax = plt.subplots(figsize=(fig_width, fig_height), facecolor=facecolor)

        # Create color array
        colors = color_gradient.to_rgba(plot_values)

        # # Set zero values to be transparent (optional - remove if you want them visible)
        # zero_mask = all_values <= 0
        # colors[zero_mask, 3] =   # Set alpha to 0 for zero values only

        zero_mask = all_values <= 0
        zero_color = np.array([0.9, 0.9, 0.9, 1.0])  # RGBA for light gray
        colors[zero_mask] = zero_color
        
        # Create EllipseCollection
        ells = EllipseCollection(widths=circle_diameter, heights=circle_diameter, angles=0, units='x',
                                 facecolors=colors,
                                 offsets=transformed_positions,
                                 transOffset=ax.transData,
                                 edgecolors='none')

        ax.add_collection(ells)

        ax.set_xlim(x_min, x_max)
        ax.set_ylim(y_min, y_max)
        ax.set_aspect('equal', adjustable='box')

        # Remove axes
        ax.axis('off')

        # Add colorbar
        divider = make_axes_locatable(ax)
        cax = divider.append_axes("right", size="5%", pad=0.1)
        cbar = plt.colorbar(color_gradient, cax=cax)
        value_label = 'Time (ns)' if plot_time else 'Photoelectron Count (a.u.)'
        scale_label = ' (log scale)' if log_scale else ''
        cbar.set_label(f'{value_label}{scale_label}', color=barcolor, fontsize=14)
        cbar.ax.yaxis.set_tick_params(color=barcolor, labelcolor=barcolor)
        cbar.outline.set_edgecolor(barcolor)

        # Explicitly set the tick formatter and locator to ensure ticks appear
        if log_scale:
            from matplotlib.ticker import LogLocator, LogFormatter
            cbar.ax.yaxis.set_major_locator(LogLocator(numticks=10))
            cbar.ax.yaxis.set_minor_locator(LogLocator(numticks=10, subs='auto'))
            cbar.ax.yaxis.set_major_formatter(LogFormatter())
        else:
            from matplotlib.ticker import MaxNLocator
            cbar.ax.yaxis.set_major_locator(MaxNLocator(nbins=6))

        # Ensure tick labels are white and visible
        cbar.ax.tick_params(axis='y', which='both', colors=barcolor, labelcolor=barcolor)
        for label in cbar.ax.get_yticklabels():
            label.set_color(barcolor)

        # Adjust layout
        plt.tight_layout()

        if file_name:
            plt.savefig(file_name, bbox_inches='tight', pad_inches=0.1, facecolor=facecolor, edgecolor='none')
        plt.show()

    return display_detector_data

In [ ]:
event_location = 'output/data_like_event.h5'

event_dir = Path('output')
event_dir.mkdir(parents=True, exist_ok=True)
output_dir = Path('events')
output_dir.mkdir(parents=True, exist_ok=True)
figures_dir = Path('figures')
figures_dir.mkdir(parents=True, exist_ok=True)

# Save the noisy data-like event
save_single_event(true_data, true_track, detector_params, filename=event_location, calibration_mode=False)

loaded_trk_params, loaded_detector_params, loaded_indices_t, loaded_charges, loaded_times_t = load_single_event(event_location, NUM_DETECTORS, calibration_mode=False)
print_particle_params(loaded_trk_params)

detector_display = create_detector_display(default_json_filename)
detector_display(loaded_indices_t, loaded_charges, loaded_times_t, file_name='figures/data_event_display_charge.pdf', plot_time=False, perc_min=0.0, log_scale=False)
detector_display(loaded_indices_t, loaded_charges, loaded_times_t, file_name='figures/data_event_display_time.pdf', plot_time=True, perc_min=0.0, perc_max=50.0)

event_location = 'output/pred_event.h5'
save_single_event(pred_data, true_track, detector_params, filename=event_location, calibration_mode=False)

loaded_trk_params, loaded_detector_params, loaded_indices, loaded_charges, loaded_times_p = load_single_event(event_location, NUM_DETECTORS, calibration_mode=False)
print_particle_params(loaded_trk_params)

detector_display(loaded_indices, loaded_charges, loaded_times_p, file_name='figures/pred_event_display_charge.pdf', plot_time=False, log_scale=False)
detector_display(loaded_indices, loaded_charges, loaded_times_p, file_name='figures/pred_event_display_time.pdf', plot_time=True, perc_min=0.0, perc_max=100.0)